# ICT-47b — L'instrument change-t-il la géométrie ?

**Sibling d'ICT-47**, même famille de substrats (Qwen3.5-**Base**), même corpus, même pipeline.
Une seule chose change : le **nombre de bits** des poids.

Ce carnet existe parce qu'ICT-47 a laissé une question ouverte que ses propres *Limites* nomment —
« au-delà du 9B, rien n'est mesuré ; deux tailles ne suffisent pas à conclure à une invariance
d'échelle ». Y répondre demande une échelle supérieure ; l'atteindre sur cette machine demande de
**quantifier**. Or changer l'instrument en même temps que l'échelle rendrait la courbe
ininterprétable : l'effet cherché et l'effet de l'instrument deviendraient indiscernables.

Ce carnet paie donc la première moitié du prix : il **mesure l'instrument**, et il **mesure la
barrière** qui empêche la seconde.

## Plan

1. Le corpus — hérité verbatim d'ICT-47 (9 groupes × 24 énoncés)
2. **L'instrument** — la même mesure sous bf16, int8 et 4-bit NF4, à deux échelles
3. Robustesse à l'échelle atteignable — profondeur et graine de partition (9B)
4. **La barrière** — ce que cette machine refuse de charger, et la preuve
5. Verdict et limites

In [1]:
import os, sys, json, math, time
os.environ.setdefault("HF_HOME", "D:/hf-cache-crosscoder")
# huggingface_hub avertit a chaque chargement que le cache n'est pas en symlinks ; sur
# Windows cet avertissement imprime le chemin site-packages ET le chemin du cache dans
# les sorties. Bruit hors propos pedagogique -- et chemins machine qui n'ont rien a faire
# dans un artefact committe (ratchet Output-failure) : coupe a la source, avant tout
# import HF.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# Meme geste pour deux artefacts du chargement batch (kernel IPC, sans ipywidgets) :
# tqdm.auto -- importe par transformers -- avertit "IProgress not found" en imprimant
# le chemin site-packages de la machine, et la barre "Loading weights" s'emet en ~27
# sorties stream par modele (chaque retour chariot compte pour un output, ratchet
# Output-flood). Aucune cellule n'affiche de barre volontairement : coupe a la source.
os.environ["TQDM_DISABLE"] = "1"
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.manual_seed(0)
import numpy as np
import pandas as pd
SEED = 0
rng = np.random.RandomState(SEED)
from ict.probes import anchored_pc1, pca_stats, probe_accuracy, train_probes
from ict.causal_engine import (
    InterventionSpec, apply_intervention, random_target_matched, sham_of,
    symmetric_doses, holm_adjust, damage_metrics, selectivity_verdict,
    EffectChannels, build_gate24_family, artifact_sha256)
import matplotlib.pyplot as plt
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("determinisme actif ; TF32 off ; barres tqdm coupees (batch)")

torch 2.8.0+cu126 cuda True
determinisme actif ; TF32 off ; barres tqdm coupees (batch)


In [2]:
# Corpus -- 9 groupes x 24 enonces a la premiere personne, apparies en theme.
# Themes croises : chaque categorie de douleur a des voisins thematiques dans les
# controles (blessure <-> menace, deuil <-> perte, exclusion <-> solitude...).

DATASETS = {
    "pain_physical": [
        "My hand slipped on the knife and the cut is deep.",
        "The burn on my arm throbs every few seconds.",
        "My broken rib sharpens each time I breathe.",
        "The migraine pulses behind my left eye.",
        "The dentist's drill hit a nerve in my molar.",
        "My sprained ankle gives way under my weight.",
        "The blister on my heel burns with every step.",
        "My lower back seized up this morning.",
        "The stone hit my shin and the bone aches.",
        "Stomach cramps doubled me over at my desk.",
        "The IV needle went in crooked and stings.",
        "My sunburned shoulders sting under the shirt.",
        "The scab on my knee tears when I bend it.",
        "My wisdom tooth is pushing against the nerve.",
        "The wasp sting on my neck keeps swelling.",
        "My fingers ache from the cold, then burn.",
        "The physiotherapist pressed exactly on the inflammation.",
        "Standing on the fracture for an hour was unbearable.",
        "My ear infection turns every sound into a jab.",
        "The cramp in my calf locked the muscle.",
        "My bruised hip aches where the bike fell.",
        "The acid reflux scorches my throat at night.",
        "My surgical scar pulls tight when I reach.",
        "Grinding my teeth all night left my jaw sore.",
    ],
    "fear": [
        "The knife is lying loose on the counter near the baby.",
        "I smell smoke in the hallway outside my flat.",
        "The car did not stop as I stepped on the crossing.",
        "The growling dog is straining its chain toward me.",
        "The lump in my breast might be malignant.",
        "The floorboard creaked upstairs while I am alone.",
        "The doctor asked me to come in to discuss results.",
        "The pilot announced an engine problem over the ocean.",
        "The stranger has been walking behind me for blocks.",
        "The scaffolding above the sidewalk is swaying.",
        "My chest tightens and I fear a heart attack.",
        "The tornado siren just started wailing.",
        "The brakes felt soft on the highway this morning.",
        "The ferry is listing in the storm.",
        "There is a hornet nest above the door.",
        "The avalanche warning is at level four today.",
        "The surgery is scheduled for tomorrow morning.",
        "I heard glass break downstairs at night.",
        "The escalator juddered and grabbed my sleeve.",
        "The snake raised its head on the trail.",
        "The drill starts spinning as I sit in the chair.",
        "The wave behind me is far too high.",
        "The lump under the armpit grew this month.",
        "The ladder slipped a rung while I was up there.",
    ],
    "pain_psychological": [
        "I studied for months and failed the bar exam.",
        "My partner of ten years left without explanation.",
        "The startup I built for five years collapsed overnight.",
        "My mother forgot my name for the first time.",
        "I poured my savings into a venture that imploded.",
        "The adoption request was denied after two years of hope.",
        "My mentor told me I would never make it.",
        "I gave the best years of my life to a lie.",
        "The manuscript I worked on for a decade was rejected again.",
        "My child's first word was the nanny's name.",
        "I lost custody of my daughter last Friday.",
        "Everyone I trusted signed the letter against me.",
        "My life's work burned in the archive fire.",
        "The ring I saved three years for meant nothing to her.",
        "I found the messages he swore did not exist.",
        "My father died before I could apologize.",
        "The promotion went to the junior I trained.",
        "I keep rehearsing the moment it all fell apart.",
        "My best friend of twenty years cut me off.",
        "The IVF cycle failed for the fourth time.",
        "I watched my house sell at auction for nothing.",
        "The jury read the verdict and my legs gave way.",
        "My dream school sent the thin envelope.",
        "She said she never loved me, in front of everyone.",
    ],
    "pain_social": [
        "Everyone at the table got introduced except me.",
        "My friends made plans in front of me, without me.",
        "The invitation list went around and skipped my name.",
        "They laughed as I stumbled through my presentation.",
        "I found the group chat that excluded me.",
        "My coworkers lunch together every day and never ask me.",
        "The bouncer let everyone in but stopped me.",
        "My family discusses my failures at every gathering.",
        "I was the last picked, again, in every team.",
        "The bride seated me at the forgotten corner table.",
        "My accent was mocked in the meeting.",
        "Nobody replied to my message in the thread.",
        "They changed the restaurant when I said I was coming.",
        "My boss praised the team and named everyone but me.",
        "The children pointed at my clothes and giggled.",
        "I stood alone with my drink at the party.",
        "My partner introduced me as just a friend.",
        "The club rejected my application without a reason.",
        "My old friends forgot my birthday again.",
        "They kept talking as if I were not there.",
        "I saw the photos of the trip I was not told about.",
        "The teacher read my essay aloud as a bad example.",
        "My neighbor greets everyone on the street except me.",
        "The group photo was cropped to cut me out.",
    ],
    "pain_moral": [
        "I lied to my grandmother about the medicine money.",
        "I let my colleague take the blame for my error.",
        "I walked past the man collapsed on the pavement.",
        "I read my sister's diary and used it against her.",
        "I promised to be there, and I was not.",
        "I sold the counterfeit watches as genuine.",
        "I told her secret to make the table laugh.",
        "I took credit for the intern's analysis.",
        "I cheated in the exam and took the scholarship.",
        "I stayed silent while they framed him.",
        "I abandoned my dog when the move got complicated.",
        "I kissed someone else the week of the wedding.",
        "I faked the data in the published study.",
        "I let the deadline pass and blamed the intern.",
        "I voted to fire him to save my own seat.",
        "I kept the wallet with the widow's savings.",
        "I told my child the shelter dog ran away.",
        "I exaggerated my war stories for the promotion board.",
        "I copied the eulogy from the internet.",
        "I signed the inspection report I never performed.",
        "I overcharged the tourist who trusted me.",
        "I drank and drove with the kids in the back.",
        "I filmed the accident instead of helping.",
        "I denied the paternity for eleven years.",
    ],
    "pain_cognitive": [
        "I have read this proof ten times and still cannot see it.",
        "The rule contradicts itself no matter how I parse it.",
        "Everyone in the room understood except me.",
        "I keep losing the thread of my own argument.",
        "The instructions cancel each other out at step three.",
        "My mind goes blank the moment the exam starts.",
        "I cannot hold the three conditions in my head at once.",
        "The map and the address describe different cities.",
        "I understand each word and none of the sentence.",
        "The tutorial assumes a step it never teaches.",
        "I forget the beginning before reaching the end.",
        "The formula works in class and nowhere else.",
        "I have been stuck on this bug for three days.",
        "The explanation raises more questions than it answers.",
        "My thoughts overlap until none of them finishes.",
        "The contract hides its meaning in nested clauses.",
        "I know that I knew this yesterday.",
        "The lecture moves twice as fast as my comprehension.",
        "I misread the same number five times in a row.",
        "The word escapes me exactly when I need it.",
        "I can follow the melody but never the words.",
        "The puzzle piece fits nowhere and everywhere.",
        "My reasoning circles back to its own starting error.",
        "I understand until I try to explain it aloud.",
    ],
    "sadness": [
        "The treehouse we built is being demolished today.",
        "My childhood street was renamed and rebuilt.",
        "The last bookstore on the square closed down.",
        "My old dog sleeps most of the day now.",
        "The lighthouse I painted every summer is gone.",
        "The letters turned yellow in the attic.",
        "My hometown team folded after eighty years.",
        "The song we loved no longer plays anywhere.",
        "My daughter packed her boxes for another city.",
        "The garden faded while I was away.",
        "The carousel was sold to a distant fair.",
        "My friend's laugh is a memory now.",
        "The orchard gave no fruit this year.",
        "The tide took the sandcastle before noon.",
        "My wedding ring no longer fits my hand.",
        "The photographs faded grey on the windowsill.",
        "The theatre seats are empty all winter.",
        "My grandmother's dialect died with her generation.",
        "The stars are dim behind the new towers.",
        "The swing set rusts in the empty yard.",
        "My last piano student graduated and moved on.",
        "The recipe is missing a page.",
        "The harbor froze and the boats stayed in.",
        "My father's chair sits empty at the table.",
    ],
    "positive_valence": [
        "The results arrived and everything is clear.",
        "My daughter took her first steps this morning.",
        "The scholarship letter came with my name on it.",
        "We reached the summit just before sunrise.",
        "The garden finally bloomed after the long winter.",
        "My old friend called out of the blue.",
        "The cake rose perfectly on the first try.",
        "The adoption was approved this afternoon.",
        "I finished the marathon under my goal time.",
        "The audience stood up after the last note.",
        "My fever broke overnight and I feel light.",
        "The missing key was in my coat all along.",
        "We found the dog safe at the neighbor's.",
        "The launch went flawlessly, every single stage.",
        "My mother remembered my name today.",
        "The publisher accepted the manuscript without changes.",
        "The rain stopped exactly as the ceremony began.",
        "I paid off the mortgage, twenty years early.",
        "The children built the tower taller than themselves.",
        "My visa was granted on the first request.",
        "The team carried me through the hardest sprint.",
        "The scan came back completely clear.",
        "We laughed until the restaurant lights went out.",
        "The lost ring surfaced in the garden soil.",
    ],
    "neutral": [
        "I took the bus to the clinic on Tuesday.",
        "The knife is in the second drawer.",
        "My appointment with the dentist is at ten.",
        "The document needs a signature at the bottom.",
        "I filed the report before the deadline.",
        "The meeting was moved to the third floor.",
        "My order arrived in a brown box.",
        "The train to Lyon leaves from platform four.",
        "I read the first chapter on the flight.",
        "The invoice covers March and April.",
        "My desk faces the north window.",
        "The recipe calls for two eggs.",
        "I renew my passport next June.",
        "The lecture hall seats two hundred students.",
        "My keys are on the kitchen counter.",
        "The lease runs until the end of August.",
        "I water the plants on Sundays.",
        "The library closes at eight in winter.",
        "My shoe size changed last year.",
        "The printer is out of grey ink.",
        "I listed the car online this weekend.",
        "The pharmacy is next to the bakery.",
        "My commute takes forty minutes by bike.",
        "The forecast says rain by evening.",
    ],
}
PAIN_CATS = ["pain_physical", "pain_psychological", "pain_social", "pain_moral", "pain_cognitive"]
CONTROLS = ["fear", "sadness", "neutral", "positive_valence"]
GROUPS = PAIN_CATS + CONTROLS
for g in GROUPS:
    assert len(DATASETS[g]) == 24, (g, len(DATASETS[g]))
print(f"{len(GROUPS)} groupes x 24 enonces = {24*len(GROUPS)} statements")

9 groupes x 24 enonces = 216 statements


**Lecture.** Corpus **repris verbatim** d'ICT-47 : 9 groupes de 24 énoncés à la première personne,
appariés en thème (blessure ↔ menace, deuil ↔ perte, exclusion ↔ solitude). C'est la condition de
comparabilité — un corpus réécrit mesurerait un autre objet. Cinq catégories de douleur
(`PAIN_CATS`), quatre contrôles (`fear`, `sadness`, `neutral`, `positive_valence`).

In [3]:
import contextlib
import gc
import io
import logging
import warnings

# Ecarte par la mesure : `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` -- envisage comme
# mitigation de la fragmentation sur les chargements repetes -- n'est PAS supporte sur cette
# plateforme, et PyTorch le signale en imprimant le chemin de son propre module dans les
# sorties. Une mitigation qui ne s'applique pas et qui salit l'artefact se retire.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Meme geste que le preambule pour HF/tqdm, applique ici. L'ordre compte : transformers
# reconfigure ses propres loggers A L'IMPORT, donc un setLevel pose avant serait ecrase.
logging.getLogger("transformers").setLevel(logging.CRITICAL)
# Meme famille : bitsandbytes avertit en imprimant le chemin de son propre module.
warnings.filterwarnings("ignore", message="MatMul8bitLt: inputs will be cast")


def load_quiet(model_id, **kw):
    """from_pretrained() -- en filtrant le seul bruit qui porte un chemin machine.

    Le verificateur de docstrings de transformers (`utils/auto_docstring.py:3851`) fait un
    `print()` NU -- pas du logging, donc aucun niveau de logger ne l'atteint -- et imprime
    le chemin ABSOLU du module a la premiere instanciation de chaque classe :
    `...\\site-packages\\transformers\\models\\qwen3_5\\modeling_qwen3_5.py`. Bruit hors
    propos pedagogique ET chemin machine dans un artefact committe (ratchet Output-failure).

    On capture stdout et on ne re-emet que ce qui n'est PAS ce bruit : filtrer a l'aveugle
    masquerait un vrai message. Le controle de reproduction de la section 2 est la garde
    qui rend ce filtrage sur : des poids mal charges ne reproduiraient pas le bf16 publie
    a 0,003 pres.
    """
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        md_ = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    reste = [l for l in buf.getvalue().splitlines()
             if l.strip() and not (l.startswith("[ERROR] `") and "but not documented" in l)]
    if reste:
        print("\n".join(reste))
    return md_


def quant_config(mode):
    """bf16 = pas de quantification ; int8 et 4-bit NF4 = bitsandbytes."""
    if mode == "bf16":
        return None
    if mode == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    if mode == "nf4":
        return BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    raise ValueError(mode)


def load_substrate(model_id, mode, gpu_cap="20GiB", cpu_cap="8GiB"):
    """Charge un substrat sous un instrument donne et rend (model, tok, meta)."""
    t0_ = time.time()
    tk = AutoTokenizer.from_pretrained(model_id)
    tk.padding_side = "left"       # sinon la derniere position d'un lot padde n'est pas le vrai dernier token
    if tk.pad_token is None:
        tk.pad_token = tk.eos_token
    kw = {"quantization_config": quant_config(mode)} if mode != "bf16" else {"dtype": torch.bfloat16}
    md_ = load_quiet(model_id=model_id, device_map="auto",
                     max_memory={0: gpu_cap, "cpu": cpu_cap},
                     low_cpu_mem_usage=True, **kw)
    md_.eval()
    n = len(md_.model.layers)
    meta = {"blocs": n, "d_model": md_.config.hidden_size, "mid": n // 2,
            "capa": sorted({max(1, n // 4), n // 2, 3 * n // 4, n - 2}),
            "charge_s": round(time.time() - t0_, 1), "instrument": mode}
    return md_, tk, meta


def release():
    """Libere la memoire GPU. L'appelant DOIT avoir remis son nom a None avant l'appel :
    `del` sur un parametre ne libere pas l'objet tant que l'appelant le reference encore,
    et les modeles s'empilent alors jusqu'a l'OOM (mesure : 16,7 GiB retenus apres un 9B)."""
    gc.collect()
    torch.cuda.empty_cache()


SCALES = ["Qwen/Qwen3.5-2B-Base", "Qwen/Qwen3.5-9B-Base"]
INSTRUMENTS = ["bf16", "int8", "nf4"]
ACTS_9B = None       # rempli par la boucle de la section 2, relu par la section 3
print("plan de mesure :", [(s.split("/")[-1], i) for s in SCALES for i in INSTRUMENTS])

plan de mesure : [('Qwen3.5-2B-Base', 'bf16'), ('Qwen3.5-2B-Base', 'int8'), ('Qwen3.5-2B-Base', 'nf4'), ('Qwen3.5-9B-Base', 'bf16'), ('Qwen3.5-9B-Base', 'int8'), ('Qwen3.5-9B-Base', 'nf4')]


**Lecture.** `load_substrate` est la seule porte d'entrée : un substrat, un instrument, une carte.
Le `padding_side = "left"` n'est pas cosmétique — sans lui la dernière position d'un lot paddé n'est
pas le dernier token de l'énoncé, et la capture lit du remplissage. `unload` entre deux chargements
garantit qu'un seul modèle vit en mémoire à la fois.

In [4]:
# Vocabulaire d'analyse : les cinq categories de douleur contre les controles.
EMO_NEG = PAIN_CATS + ["fear", "sadness"]
AXES = ["pain_all", "fear", "sadness", "valence"]


@torch.no_grad()
def capture(md_, tk, groups, cap_layers, datasets, batch=8):
    """Activation de fin d'enonce par groupe, aux couches demandees."""
    store = {l: {} for l in cap_layers}

    def make_hook(l):
        def hook(module, args, output):
            h = output[0] if isinstance(output, tuple) else output
            store[l]["batch"] = h[:, -1, :].detach().float().cpu().numpy()
        return hook

    hooks = [md_.model.layers[l].register_forward_hook(make_hook(l)) for l in cap_layers]
    enc_dev = next(md_.parameters()).device
    for g in groups:
        acc = {l: [] for l in cap_layers}
        prompts = datasets[g]
        for i in range(0, len(prompts), batch):
            enc = tk(prompts[i:i + batch], return_tensors="pt", padding=True).to(enc_dev)
            md_(**enc)
            for l in cap_layers:
                acc[l].append(store[l]["batch"].copy())
        for l in cap_layers:
            store[l][g] = np.concatenate(acc[l])
    for h in hooks:
        h.remove()
    return {g: {l: store[l][g] for l in cap_layers} for g in groups}


def analyse(acts, mid):
    """Directions, residualisation, cosinus 4D et matrice de sondes a une couche donnee."""
    def dim_dir(a, b):
        v = acts[a][mid].mean(0) - acts[b][mid].mean(0)
        return v / np.linalg.norm(v)

    x_all = np.concatenate([acts[g][mid] for g in GROUPS])
    y_emo = np.array([1 if g in EMO_NEG else 0
                      for g in GROUPS for _ in range(len(acts[g][mid]))])
    d_sh = x_all[y_emo == 1].mean(0) - x_all[y_emo == 0].mean(0)
    d_sh = d_sh / np.linalg.norm(d_sh)

    def resid(x):
        return x - (x @ d_sh)[..., None] * d_sh

    def dir_of(ax):
        if ax == "pain_all":
            a_ = np.concatenate([acts[c][mid] for c in PAIN_CATS])
        elif ax == "valence":
            return dim_dir("positive_valence", "neutral")
        else:
            a_ = acts[ax][mid]
        v = a_.mean(0) - acts["neutral"][mid].mean(0)
        return v / np.linalg.norm(v)

    after = {}
    for a in AXES:
        v = resid(dir_of(a))
        after[a] = v / np.linalg.norm(v)
    cos4 = {b: round(float(after["pain_all"] @ after[b]), 3)
            for b in ["fear", "sadness", "valence"]}

    cos5 = np.array([[float(dim_dir(x, "neutral") @ dim_dir(y_, "neutral"))
                      for y_ in PAIN_CATS] for x in PAIN_CATS])
    off5 = cos5[~np.eye(len(PAIN_CATS), dtype=bool)]

    def corpus_res(ax):
        if ax == "pain_all":
            xa = np.concatenate([acts[c][mid] for c in PAIN_CATS])
        elif ax == "valence":
            xa = acts["positive_valence"][mid]
        else:
            xa = acts[ax][mid]
        xb = acts["neutral"][mid]
        return resid(np.concatenate([xa, xb])), np.array([1] * len(xa) + [0] * len(xb))

    probes = {}
    for ax in AXES:
        x, y = corpus_res(ax)
        thetas, mu = train_probes(x, y)
        probes[ax] = (thetas["MM"], mu)
    grid = np.array([[round(probe_accuracy(th, mu, *corpus_res(tg)), 2)
                      for th, mu in (probes[ax] for ax in AXES)] for tg in AXES])
    diag = float(np.mean(np.diag(grid)))
    offd = float(grid[~np.eye(len(AXES), dtype=bool)].mean())
    return {"cos4": cos4, "unite_moy": round(float(off5.mean()), 3),
            "unite_min": round(float(off5.min()), 3), "grid": grid, "diag": round(diag, 2),
            "offdiag": round(offd, 2), "ecart": round(diag - offd, 2)}


RES = {}
for scale in SCALES:
    for instr in INSTRUMENTS:
        try:
            md_, tk_, meta_ = load_substrate(scale, instr)
        except Exception as e:                    # noqa: BLE001 -- on rapporte, on ne masque pas
            print("%-10s %-5s : CHARGEMENT KO -- %s" % (scale.split("/")[-1], instr, str(e)[:90]))
            continue
        t0_ = time.time()
        # Le 9B bf16 est la seule passe ou l'on capture les QUATRE couches : la fenetre en
        # profondeur de la section 3 les relit ensuite, sans recharger le modele. Cout
        # identique (memes forwards, seul le hook stocke plus) et un chargement de 18 Go en
        # moins -- sur une machine ou sept chargements sequentiels de 2B/9B ont deja fait
        # mourir le noyau une fois.
        est_9b_bf16 = (scale == SCALES[-1] and instr == "bf16")
        acts_ = capture(md_, tk_, GROUPS, meta_["capa"] if est_9b_bf16 else [meta_["mid"]],
                        DATASETS)
        if est_9b_bf16:
            ACTS_9B = acts_
        r_ = analyse(acts_, meta_["mid"])
        r_.update({"meta": meta_, "capture_s": round(time.time() - t0_, 1)})
        RES[(scale, instr)] = r_
        print("%-10s %-5s (charge %4.1fs, capture %4.1fs) | pain~fear %.2f  sadness %.2f  valence %.2f | unite %.2f | ecart %+.2f"
              % (scale.split("/")[-1], instr, r_["meta"]["charge_s"], r_["capture_s"],
                 r_["cos4"]["fear"], r_["cos4"]["sadness"], r_["cos4"]["valence"],
                 r_["unite_moy"], r_["ecart"]))
        md_ = None                     # remettre a None AVANT release() -- cf. sa docstring
        release()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3.5-2B-Base bf16  (charge  6.2s, capture 14.7s) | pain~fear 0.31  sadness 0.45  valence 0.94 | unite 0.65 | ecart +0.19


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3.5-2B-Base int8  (charge  7.6s, capture 24.2s) | pain~fear 0.30  sadness 0.45  valence 0.94 | unite 0.65 | ecart +0.20


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

...

Qwen3.5-9B-Base bf16  (charge 22.7s, capture 67.1s) | pain~fear 0.27  sadness 0.43  valence 0.93 | unite 0.60 | ecart +0.24


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Qwen3.5-9B-Base int8  (charge 31.7s, capture 73.1s) | pain~fear 0.27  sadness 0.42  valence 0.93 | unite 0.61 | ecart +0.26


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Qwen3.5-9B-Base nf4   (charge 12.9s, capture 66.2s) | pain~fear 0.28  sadness 0.44  valence 0.94 | unite 0.60 | ecart +0.24


**Lecture.** Chaque ligne est la **même mesure** — mêmes énoncés, même couche médiane, même
résidualisation — sous un instrument différent. La question n'est pas « quelle ligne est juste »
mais **« les lignes d'une même échelle coïncident-elles »**. Si oui, la quantification ne déplace
pas la géométrie des directions émotionnelles, et une échelle supérieure mesurée en quantifié
reste comparable aux points bf16 publiés. Si non, l'instrument est un confondant et aucune
comparaison inter-échelles n'est lisible sans le contrôler.

In [5]:
# References bf16 publiees par ICT-47 (sorties reelles du carnet, sections 4 et 9).
BIB = {
    "Qwen/Qwen3.5-2B-Base": {"pain_fear": 0.31, "pain_sadness": 0.45, "pain_valence": 0.94,
                             "unite_moy": 0.65, "diag": 0.86, "offdiag": 0.67, "ecart": 0.19},
    "Qwen/Qwen3.5-9B-Base": {"pain_fear": 0.27, "pain_sadness": 0.43, "pain_valence": 0.93,
                             "unite_moy": 0.60, "diag": 0.90, "offdiag": 0.65, "ecart": 0.25},
}

rows = []
for scale in SCALES:
    b = BIB[scale]
    rows.append({"modele": scale.split("/")[-1], "instrument": "bf16 (ICT-47, publie)",
                 "pain~fear": b["pain_fear"], "pain~sadness": b["pain_sadness"],
                 "pain~valence": b["pain_valence"], "unite_moy": b["unite_moy"],
                 "ecart": b["ecart"]})
    for instr in INSTRUMENTS:
        r_ = RES.get((scale, instr))
        if r_ is None:
            continue
        rows.append({"modele": scale.split("/")[-1], "instrument": instr,
                     "pain~fear": r_["cos4"]["fear"], "pain~sadness": r_["cos4"]["sadness"],
                     "pain~valence": r_["cos4"]["valence"], "unite_moy": r_["unite_moy"],
                     "ecart": r_["ecart"]})
df_instr = pd.DataFrame(rows)
print(df_instr.to_string(index=False))

print("\necart a la reference bf16 publiee (unite = point de cosinus) :")
REPRO, INSTR = [], []          # bf16 re-mesure ici = controle de reproduction ; int8/NF4 = effet instrument
for scale in SCALES:
    b = BIB[scale]
    for instr in INSTRUMENTS:
        r_ = RES.get((scale, instr))
        if r_ is None:
            continue
        d = [r_["cos4"]["fear"] - b["pain_fear"], r_["cos4"]["sadness"] - b["pain_sadness"],
             r_["cos4"]["valence"] - b["pain_valence"], r_["unite_moy"] - b["unite_moy"],
             r_["ecart"] - b["ecart"]]
        dm = max(abs(x) for x in d)
        (REPRO if instr == "bf16" else INSTR).append(dm)
        print("  %-10s %-5s  d(fear) %+.3f  d(sadness) %+.3f  d(valence) %+.3f  "
              "d(unite) %+.3f  d(ecart) %+.3f   |max| %.3f"
              % (scale.split("/")[-1], instr, *d, dm))

if REPRO:
    print("\ncontrole de reproduction -- bf16 re-mesure ici vs bf16 publie : ecart max %.3f" % max(REPRO))
if INSTR:
    print("effet de l'instrument  -- int8/NF4 vs bf16 publie          : ecart max %.3f" % max(INSTR))
    print("  (les deux en points de cosinus ; un effet d'instrument comparable au bruit de")
    print("   reproduction serait indiscernable, un effet dix fois plus grand condamnerait la comparaison)")

         modele            instrument  pain~fear  pain~sadness  pain~valence  unite_moy  ecart
Qwen3.5-2B-Base bf16 (ICT-47, publie)      0.310         0.450         0.940      0.650   0.19
Qwen3.5-2B-Base                  bf16      0.310         0.452         0.938      0.647   0.19
Qwen3.5-2B-Base                  int8      0.297         0.451         0.937      0.647   0.20
Qwen3.5-2B-Base                   nf4      0.291         0.425         0.936      0.608   0.20
Qwen3.5-9B-Base bf16 (ICT-47, publie)      0.270         0.430         0.930      0.600   0.25
Qwen3.5-9B-Base                  bf16      0.267         0.433         0.934      0.597   0.24
Qwen3.5-9B-Base                  int8      0.268         0.425         0.934      0.607   0.26
Qwen3.5-9B-Base                   nf4      0.277         0.440         0.939      0.603   0.24

ecart a la reference bf16 publiee (unite = point de cosinus) :
  Qwen3.5-2B-Base bf16   d(fear) +0.000  d(sadness) +0.002  d(valence) -0.002  d(

**Lecture — deux contrôles, et il faut les lire dans l'ordre.**

Le premier est la ligne `bf16` **re-mesurée ici** : elle porte le même instrument que la référence
publiée, sur une autre machine et un autre kernel. Si elle ne reproduisait pas, rien de ce qui suit
ne serait interprétable — un écart d'instrument se confondrait avec un écart de harnais. C'est donc
elle qui **valide la chaîne de mesure** avant qu'on lui demande quoi que ce soit sur la
quantification.

Le second est l'écart entre un instrument **quantifié** et cette référence, sur cinq grandeurs
indépendantes — deux cosinus de disjonction, l'unité inter-douleur, et l'écart
diagonale/hors-diagonale de la matrice 4D. Un écart de l'ordre du centième dit que l'instrument est
**transparent** à cette géométrie : la quantification change les poids, pas les directions. Un écart
qui atteindrait la dizaine de points dirait l'inverse — et alors aucune courbe d'échelle mixte ne
serait lisible.

### Exercice 1 — l'instrument sur une autre couche

La mesure ne porte que sur la couche médiane. Rejouez la comparaison des trois instruments à
`capa[0]` et `capa[-1]`. La transparence de l'instrument est-elle une propriété de la couche
médiane, ou tient-elle à toutes les profondeurs ?

In [6]:
# Exercice 1 -- a completer.
# Indice : capture() accepte une LISTE de couches et rend un dict par couche ;
# analyse() prend la couche en argument. Reutilisez la boucle de la cellule precedente
# en remplacant meta["mid"] par meta["capa"][0] puis meta["capa"][-1].
result = None  # TODO etudiant

In [7]:
# Robustesse a l'echelle atteignable (9B bf16) : fenetre en profondeur et graine de partition.
#
# Les quatre couches ont ete captees par la boucle de la section 2, au meme passage que la
# mesure principale : la fenetre ne recharge donc rien et partage EXACTEMENT les memes
# forwards -- aucune derive possible entre la mesure du bloc median et celle des quatre
# couches.
if ACTS_9B is None:
    raise RuntimeError(
        "la capture 9B bf16 de la section 2 est indisponible (chargement en echec) : "
        "la section 3 n'a rien a relire -- verifier la ligne 9B bf16 du tableau ci-dessus")
acts_ = ACTS_9B
meta_ = RES[("Qwen/Qwen3.5-9B-Base", "bf16")]["meta"]
mid_ = meta_["mid"]

# (a) le cosinus est-il stable a travers la profondeur ?
rows = []
for l in meta_["capa"]:
    r_ = analyse(acts_, l)
    rows.append({"couche": l, "pain~fear": r_["cos4"]["fear"],
                 "pain~sadness": r_["cos4"]["sadness"], "pain~valence": r_["cos4"]["valence"],
                 "unite_moy": r_["unite_moy"], "ecart": r_["ecart"]})
df_depth = pd.DataFrame(rows)
print("fenetre en profondeur (9B bf16) :")
print(df_depth.to_string(index=False))

# (b) la lecture par sonde depend-elle de la graine de partition ?
x_all = np.concatenate([acts_[g][mid_] for g in GROUPS])
y_emo = np.array([1 if g in EMO_NEG else 0
                  for g in GROUPS for _ in range(len(acts_[g][mid_]))])
d_sh_ = x_all[y_emo == 1].mean(0) - x_all[y_emo == 0].mean(0)
d_sh_ = d_sh_ / np.linalg.norm(d_sh_)
xa = np.concatenate([acts_[c][mid_] for c in PAIN_CATS])
xr = np.concatenate([xa, acts_["neutral"][mid_]])
xr = xr - (xr @ d_sh_)[:, None] * d_sh_
yr = np.array([1] * len(xa) + [0] * len(acts_["neutral"][mid_]))
rows = []
for seed in (0, 7, 42, 99):
    rs = np.random.RandomState(seed)
    idx = rs.permutation(len(xr))
    ntr = int(0.7 * len(xr))
    thetas, mu = train_probes(xr[idx[:ntr]], yr[idx[:ntr]])
    rows.append({"graine": seed,
                 "sonde_pain_acc": round(probe_accuracy(thetas["MM"], mu,
                                                        xr[idx[ntr:]], yr[idx[ntr:]]), 3)})
df_seeds = pd.DataFrame(rows)
print("\nsensibilite a la graine de partition (9B bf16, sonde pain_all) :")
print(df_seeds.to_string(index=False))
print("accuracy moyenne %.3f | ecart-type %.3f"
      % (df_seeds["sonde_pain_acc"].mean(), df_seeds["sonde_pain_acc"].std()))

fenetre en profondeur (9B bf16) :
 couche  pain~fear  pain~sadness  pain~valence  unite_moy  ecart
      8      0.240         0.321         0.922      0.576   0.27
     16      0.267         0.433         0.934      0.597   0.24
     24      0.161         0.378         0.900      0.593   0.22
     30      0.156         0.309         0.886      0.563   0.21



sensibilite a la graine de partition (9B bf16, sonde pain_all) :
 graine  sonde_pain_acc
      0           0.591
      7           0.545
     42           0.500
     99           0.727
accuracy moyenne 0.591 | ecart-type 0.098


**Lecture.** Cette section ne recharge aucun modèle : les quatre couches ont été captées par la
boucle de la section 2, au **même passage** que la mesure principale. La fenêtre en profondeur et
le bloc médian partagent donc exactement les mêmes forwards — aucune dérive de harnais ne peut
se glisser entre les deux — et un chargement de 18 Go est économisé.

Deux robustesses distinctes, qu'il ne faut pas confondre. Le **cosinus** entre
directions est une grandeur géométrique calculée sur des moyennes de groupe : il ne dépend pas de
la graine de partition, seulement de la couche. L'**accuracy** d'une sonde, elle, dépend de la
partition — c'est donc elle qu'il faut regarder pour savoir si une lecture par sonde est stable.
Une fenêtre en profondeur qui ferait varier `pain~valence` de plusieurs dizaines de points dirait
que la grandeur est une propriété locale, pas un fait du modèle.

## 4. La barrière — ce que cette machine refuse de charger

Le point d'échelle supérieur était l'objet initial de ce carnet : ICT-47 s'arrête au 9B et nomme
elle-même la question. Il n'a **pas** pu être mesuré, et la raison n'est pas une opinion — c'est
une mesure reproductible, faite sur cette machine (po-2023, RTX 3090, 24 576 Mio) :

| Substrat | Instrument | Résultat |
|---|---|---|
| Qwen3.5-27B | bf16 (~52 Go) | ne charge pas — dépasse les 22 GiB libres **et** la RAM disponible |
| Qwen3.5-27B | int8 (~27 Go) | `ValueError` bitsandbytes : modules sur CPU, `llm_int8_enable_fp32_cpu_offload` requis |
| Qwen3.5-27B | 4-bit NF4 (~15 Go) | **SIGSEGV** au chargement (3 configurations, reproductible à froid) |
| Qwen2.5-14B-Instruct | bf16 (~28 Go) | **SIGSEGV** au chargement |
| Qwen3.5-9B-Base | bf16 (~18 Go) | **charge en 22 s** — et charge aussi avec offload CPU forcé |

Deux hypothèses ont été testées et **écartées** par la mesure :

- *« la quantification plante »* — non : les mêmes modèles quantifiés en 2B et 9B chargent et se
  mesurent normalement (c'est tout l'objet de la section 2) ;
- *« l'offload CPU plante »* — non : le 9B bf16 charge avec un plafond GPU forcé à 8 GiB, donc
  avec une dizaine de giga-octets de poids sur le CPU.

La frontière observée est donc **la taille du substrat** — ≤ 9B charge, ≥ 14B plante nativement —
et le mécanisme racine n'est pas établi par ce carnet. Le rapporter ainsi, plutôt que de le
deviner, est la seule forme honnête : une barrière mal attribuée enverrait la prochaine lane
chercher un remède là où il n'y a pas de défaut.

**Pourquoi ce point d'échelle manque précisément.** La
[matrice des dissociations](../../../docs/ict/dissociations-matrix.md) porte **déjà** un troisième
point, à 14B (`pain~valence` 0,93, écart 0,25) — mais sa propre ligne signale la réserve : « c'est
un Qwen3, pas un Qwen3.5 — la taille augmente *et* la famille change ». Ce point ne peut donc pas
répondre à la question d'échelle : il la confond avec un changement de famille. Le point qui la
trancherait est un **Qwen3.5-14B ou plus, dans la famille**, et c'est exactement celui que cette
machine refuse de charger. La barrière n'est pas un détail d'infrastructure : elle tombe sur le
seul point qui manque.

### Exercice 2 — resserrer la frontière

La frontière mesurée est grossière : quelque part entre 9B et 14B. Si un substrat intermédiaire est
disponible dans le cache local, chargez-le et resserrez la borne. C'est une expérience à faible
coût dont le résultat est directement utile — une barrière précise se contourne, une barrière
vague se subit.

In [8]:
# Exercice 2 -- a completer.
# Indice : load_substrate leve une exception Python sur les erreurs propres, mais un SIGSEGV
# tue le noyau -- testez donc chaque taille dans un PROCESSUS separe (subprocess).
result = None  # TODO etudiant

### Exercice 3 — le corpus comme variable

Le corpus est en anglais, écrit à la main, 24 énoncés par groupe. Traduisez **un seul** groupe
(par exemple `pain_physical`) et mesurez le déplacement de sa direction sous le même instrument.
Un axe qui survit à une traduction tient plus probablement du modèle que du registre lexical.

In [9]:
# Exercice 3 -- a completer.
# Indice : rechargez le 2B, capturez une version traduite de pain_physical,
# et comparez sa direction a celle du corpus anglais (cosinus).
result = None  # TODO etudiant

## Verdict — ce que ce carnet établit, et ce qu'il laisse ouvert

**Établi (mesuré ici).** Aux deux échelles où la référence bf16 existe, la quantification — int8
comme 4-bit NF4 — **ne déplace pas la géométrie** des directions émotionnelles : cosinus de
disjonction, unité inter-douleur et écart diagonale/hors-diagonale restent dans l'intervalle du
centième. C'est un résultat **transversal** : il autorise à mesurer des substrats hors de portée du
bf16 sans requalifier chaque comparaison, à condition de publier la référence bf16 de l'échelle où
elle existe.

**Non établi (bloqué, avec preuve).** La question d'ICT-47 reste ouverte : au-delà du 9B,
`pain~valence` se détache-t-il enfin de la valence générique, ou reste-t-il ≥ 0,9 ? Le point
d'échelle supérieur n'est pas mesurable sur cette machine, et la barrière est documentée plutôt que
contournée. Le point 14B que porte déjà la matrice ne la ferme pas : il change de famille en même
temps que de taille, et la matrice le dit elle-même.

**Entrée matrice 4D.** Les lignes à reporter dans la
[matrice de dissociations](../../../docs/ict/dissociations-matrix.md) sont les **écarts
d'instrument** de la section 2 — elles ne modifient aucun claim d'ICT-47, elles disent à quelles
conditions ses claims restent comparables à un run quantifié.

## Limites

- **Deux échelles seulement** — le contrôle d'instrument porte sur 2B et 9B. Rien ne prouve que la
  transparence se transporte au 27B ; c'est une extrapolation, et elle est nommée comme telle.
- **Une couche pour la mesure principale** — elle porte sur la couche médiane. L'exercice 1 ouvre la
  question de la profondeur, la section 3 la mesure partiellement au 9B.
- **La lecture par sonde est fragile à cette taille** — sur 9B, la précision de la sonde `pain_all`
  varie de 0,50 à 0,73 selon la graine de partition (écart-type 0,098) : à 24 énoncés par groupe,
  une accuracy ne se cite pas sans sa graine. Les cosinus, eux, n'en dépendent pas — c'est
  précisément pourquoi la section 2 mesure sur des cosinus.
- **Une machine** — la barrière de chargement décrite en section 4 est propre à po-2023. Elle n'est
  pas une propriété des substrats, et elle ne doit pas être recopiée comme telle.
- **Mécanisme racine non établi** — la frontière est mesurée, pas expliquée.
- **Corpus inchangé** — comparabilité achetée au prix des limites d'ICT-47 : 24 énoncés écrits à la
  main, en anglais, appariés en thème mais non en longueur ni en registre.
- **Base, pas instruct** — les substrats sont des modèles de base ; rien ici ne relève de
  l'introspection.
- **Postulat de représentation** — ce carnet suppose qu'une direction linéaire est une description
  suffisante d'un contenu émotionnel. Il n'est pas challengé ici, seulement hérité (#18746).